# Download Market Data — CryptoStatArb

**One place to pull the project's canonical dataset.** Run this notebook once (or whenever you
want fresh data); every other notebook just loads the pickle it writes — no re-fetching.

- **Source:** Binance.US (`python-binance`, public klines — no API key needed).
- **Universe:** 18 USDT pairs. **Interval:** 1h. **History:** ~3 years.
- **Timestamps:** UTC (matters for midnight-aligned resampling downstream).

**Outputs** (in `data/`, gitignored — regenerate by re-running):

| file | shape | use |
|---|---|---|
| `binance_us_hourly_ohlcv.pk` | `(time, [Field × Ticker])` | full Open/High/Low/Close/Volume panel |
| `binance_us_hourly_close.pk` | `(time, Ticker)` | close-only convenience view |

**Load in any notebook:**
```python
import pandas as pd
panel = pd.read_pickle('data/binance_us_hourly_ohlcv.pk')
close = panel['Close']              # or: pd.read_pickle('data/binance_us_hourly_close.pk')
volume = panel['Volume']
```

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from binance.client import Client

client = Client(tld='US')   # Binance.US public endpoints
print('python-binance ready')

In [ ]:
# --- Config: edit here to change what gets pulled ---
UNIVERSE = ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'ADAUSDT', 'XRPUSDT', 'XLMUSDT',
            'DOGEUSDT', 'LINKUSDT', 'AVAXUSDT', 'SOLUSDT', 'DOTUSDT', 'LTCUSDT',
            'BCHUSDT', 'UNIUSDT', 'ATOMUSDT', 'NEARUSDT', 'ICPUSDT', 'SHIBUSDT']
INTERVAL = '1h'
START = '2023-08-27'          # ~3 years back; use e.g. '2020-01-01' for max history

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
OHLCV_PK = DATA_DIR / 'binance_us_hourly_ohlcv.pk'
CLOSE_PK = DATA_DIR / 'binance_us_hourly_close.pk'
FORCE_REFRESH = False        # True = re-download even if the pickle already exists

In [ ]:
def fetch_ohlcv(symbol, interval=INTERVAL, start=START):
    """Full OHLCV for one symbol as a UTC-indexed DataFrame (columns Open/High/Low/Close/Volume)."""
    raw = client.get_historical_klines(symbol, interval, start)
    cols = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
            'quote_volume', 'num_trades', 'taker_base_volume', 'taker_quote_volume', 'ignore']
    df = pd.DataFrame(raw, columns=cols)
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    df = df.set_index('open_time')[['open', 'high', 'low', 'close', 'volume']].astype(float)
    df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    return df

In [ ]:
# --- Download (or load cache) and save the canonical pickles ---
if OHLCV_PK.exists() and not FORCE_REFRESH:
    panel = pd.read_pickle(OHLCV_PK)
    print('Loaded cached panel (set FORCE_REFRESH=True to re-pull).')
else:
    frames = {}
    for i, s in enumerate(UNIVERSE, 1):
        t0 = time.time()
        frames[s] = fetch_ohlcv(s)
        print(f'[{i}/{len(UNIVERSE)}] {s}: {len(frames[s])} bars ({time.time()-t0:.1f}s)')
        time.sleep(0.2)                       # be gentle on the API
    panel = pd.concat(frames, axis=1).swaplevel(axis=1).sort_index(axis=1)  # -> (Field, Ticker)
    panel = panel.reindex(columns=['Open', 'High', 'Low', 'Close', 'Volume'], level=0)
    panel.columns.names = ['Field', 'Ticker']
    panel.index.name = 'open_time_utc'
    panel.to_pickle(OHLCV_PK)
    panel['Close'].to_pickle(CLOSE_PK)         # close-only convenience view
    print(f'\nSaved {OHLCV_PK.name} and {CLOSE_PK.name}')

print(panel.shape, '|', panel.index.min(), '->', panel.index.max())

In [ ]:
# --- Sanity checks ---
print('Fields :', list(panel.columns.get_level_values('Field').unique()))
print('Tickers:', list(panel.columns.get_level_values('Ticker').unique()))

coverage = (1 - panel['Close'].isna().mean()).sort_values()
print('\nClose coverage (lowest 3):', {k: round(v, 3) for k, v in coverage.head(3).items()})
panel['Close'].tail()